# Hvordan behandle tall


## Introduksjon: Python filtyper og Pandas
### Hva er Pandas?
Pandas er et bibliotek i Python som brukes til å jobbe med data.
Det er spesielt nyttig for:

lese inn data fra filer (CSV, Excel)
rydde og strukturere data
analysere tidsserier (som kraftforbruk)
gjøre beregninger og statistikk

Den viktigste datastrukturen i pandas er:

DataFrame → en tabell med rader og kolonner (som Excel)

Vi legger først inn data i en egen folder som vi kaller 'data'

og deretter leser vi den i pandas.

Når vi skriver det på denne måten:

df = pd.read_csv("../data/load_data.csv")

../ betyr en mappe opp

In [1]:
import pandas as pd

df = pd.read_csv("../data/load_data.csv", sep=",")
df

,Time(Local),Production,Consumption
0,01.01.2026 00:00:00 +01:00,"18448,59","16662,95"
1,01.01.2026 01:00:00 +01:00,"18693,66","14953,24"
2,01.01.2026 02:00:00 +01:00,"18639,68","14298,07"
3,01.01.2026 03:00:00 +01:00,"18502,66","13254,81"
4,01.01.2026 04:00:00 +01:00,"18458,45","12678,51"
...,...,...,...
3354,20.05.2026 19:00:00 +02:00,"13799,35","19331,51"
3355,20.05.2026 20:00:00 +02:00,"13857,89","20096,14"
3356,20.05.2026 21:00:00 +02:00,"13768,33","19944,19"
3357,20.05.2026 22:00:00 +02:00,"13646,36","19368,15"


In [2]:
print(df.columns)

Index(['Time(Local)', 'Production', 'Consumption'], dtype='str')


🔹 Hva er en tidsserie?
I kraftsystemer jobber vi ofte med tidsserier:
👉 data som er målt over tid
Eksempel:

last (MW)

produksjon

frekvens

## Viktig: DatetimeIndex
For å analysere tidsserier setter vi tid som indeks:

Selv med parse_dates, kan pandas feile hvis formatet er rart.

In [3]:
df = pd.read_csv("load_data.csv", parse_dates=["Time(Local)"])
#df = df.set_index("Time(Local)")

In [4]:
print(df["Time(Local)"].head())
print(type(df["Time(Local)"].iloc[0]))

0    01.01.2026 00:00:00 +01:00
1    01.01.2026 01:00:00 +01:00
2    01.01.2026 02:00:00 +01:00
3    01.01.2026 03:00:00 +01:00
4    01.01.2026 04:00:00 +01:00
Name: Time(Local), dtype: str
<class 'str'>


In [5]:
print(df.columns)

Index(['Time(Local)', 'Production', 'Consumption'], dtype='str')


In [6]:

df.columns = df.columns.str.strip()

df["Time(Local)"] = pd.to_datetime(
    df["Time(Local)"],
    dayfirst=True,
    utc=True   # ✅ THIS solves your error
)

df = df.set_index("Time(Local)")
df = df.sort_index()

df = df.tz_convert("Europe/Oslo")


In [7]:
print(df.columns)

Index(['Production', 'Consumption'], dtype='str')


In [8]:
print(df.head())

                          Production Consumption
Time(Local)                                     
2026-01-01 00:00:00+01:00   18448,59    16662,95
2026-01-01 01:00:00+01:00   18693,66    14953,24
2026-01-01 02:00:00+01:00   18639,68    14298,07
2026-01-01 03:00:00+01:00   18502,66    13254,81
2026-01-01 04:00:00+01:00   18458,45    12678,51


Første linje i dataene 18448,59 er norsk format: , = desimaltegn

pandas har lest det som tekst

FIX (konverter til tall)

In [9]:
df["Consumption"] = df["Consumption"].str.replace(",", ".").astype(float)

In [10]:
print(df.dtypes)

Production         str
Consumption    float64
dtype: object


Da kan vi gjøre:

In [14]:
analyse = df['Consumption'].resample("1D").mean()
analyse

Time(Local)
2026-01-01 00:00:00+01:00    15464.587083
2026-01-02 00:00:00+01:00    19360.323333
2026-01-03 00:00:00+01:00    23015.297083
2026-01-04 00:00:00+01:00    23660.607500
2026-01-05 00:00:00+01:00    25149.064167
                                 ...     
2026-05-16 00:00:00+02:00    13329.930833
2026-05-17 00:00:00+02:00    12752.329583
2026-05-18 00:00:00+02:00    15963.636250
2026-05-19 00:00:00+02:00    15252.570833
2026-05-20 00:00:00+02:00    15242.741667
Freq: D, Name: Consumption, Length: 140, dtype: float64

🧠 Hvorfor dette er viktig i dette kurset
I dette kurset bruker vi pandas til å:
✅ lese ekte kraftsystemdata
✅ analysere lastprofiler
✅ finne mønstre i forbruk
✅ koble data til dynamiske modeller

🎯 Kobling videre
Når dataene er klare, kan vi:

lage en representativ døgnprofil
glatte data (Gaussian-filter)
analysere dynamikk (swing equation)

In [16]:
import numpy as np

# df: time-serie med load
# index = datetime

# Lag daglig matrise (hver dag = 24 verdier)
df['date'] = df.index.date
daily = df.groupby('date')['Consumption'].apply(list)
print('Daglige tidsserier:',daily)
# Fjern dager som ikke har 24 timer
daily = daily[daily.apply(len) == 24]
print(daily)
# Konverter til array
X = np.vstack(daily.values)
print(X)
# Gjennomsnittsdøgn
mean_profile = X.mean(axis=0)
print(mean_profile)
# Finn dag med minst avvik (minste norm)
distances = np.linalg.norm(X - mean_profile, axis=1)

# Velg mest representative dag
idx = np.argmin(distances)

rep_profile = X[idx]
rep_date = daily.index[idx]


print("Representativ dag:", rep_date)

Daglige tidsserier: date
2026-01-01    [16662.95, 14953.24, 14298.07, 13254.81, 12678...
2026-01-02    [14821.37, 13826.33, 12743.11, 12786.86, 12647...
2026-01-03    [22115.9, 21572.64, 20710.6, 20344.44, 19724.8...
2026-01-04    [23251.09, 22995.68, 22691.57, 22255.31, 21886...
2026-01-05    [24208.62, 23689.68, 23986.93, 23956.24, 24106...
                                    ...                        
2026-05-16    [16276.98, 14535.05, 13984.31, 13857.01, 13656...
2026-05-17    [14220.04, 12912.59, 12540.78, 12134.04, 11770...
2026-05-18    [15629.79, 14382.06, 14139.4, 14017.26, 14589....
2026-05-19    [17748.79, 17015.92, 16436.45, 15904.32, 15931...
2026-05-20    [14881.17, 14265.45, 13891.66, 13679.98, 13776...
Name: Consumption, Length: 140, dtype: object
date
2026-01-01    [16662.95, 14953.24, 14298.07, 13254.81, 12678...
2026-01-02    [14821.37, 13826.33, 12743.11, 12786.86, 12647...
2026-01-03    [22115.9, 21572.64, 20710.6, 20344.44, 19724.8...
2026-01-04    [23251.09, 229

In [ ]:

import matplotlib.pyplot as plt

t = np.arange(24)

plt.plot(t, rep_profile, linewidth=3)
plt.xlabel("Time")
plt.ylabel("MW")
plt.title(f"Representativ dag: {rep_date.strftime('%d.%m.%Y')}")
plt.grid(True)
plt.show()


In [ ]:

from scipy.ndimage import gaussian_filter1d

smooth = gaussian_filter1d(rep_profile, sigma=2)


In [ ]:

plt.plot(t, rep_profile, label="Rådata", alpha=0.5)
plt.plot(t, smooth, label="Glattet", linewidth=3)

plt.legend()
plt.grid(True)
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import numpy as np

t = np.arange(24)

plt.plot(t, rep_profile, label="Rådata", alpha=0.5)
plt.plot(t, smooth, label="Glattet (sigma=2)", linewidth=3)

plt.xlabel("Time")
plt.ylabel("MW")
plt.title("Glatting av lastprofil")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

for s in [1, 2, 4]:
    plt.plot(gaussian_filter1d(rep_profile, sigma=s), label=f"sigma={s}")
plt.plot(t, rep_profile, label="Rådata", alpha=0.5)
plt.legend()
plt.title("Effekt av sigma")
plt.grid(True)
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from IPython.display import clear_output
import time

t = np.arange(24)

# bruk din representativ profil
data = rep_profile.copy()

plt.figure(figsize=(8,5))

# animasjon: øk sigma gradvis
for sigma in np.linspace(0.1, 5, 30):
    clear_output(wait=True)

    smooth = gaussian_filter1d(data, sigma=sigma)

    plt.clf()
    plt.plot(t, data, label="Rådata", alpha=0.4)
    plt.plot(t, smooth, label=f"Smoothing (sigma={sigma:.2f})", linewidth=3)

    plt.xlabel("Time")
    plt.ylabel("MW")
    plt.title("Hvordan sigma påvirker glattning")
    plt.legend()
    plt.grid(True)

    plt.pause(0.1)


In [ ]:
from ipywidgets import interact

def plot_sigma(sigma):
    smooth = gaussian_filter1d(rep_profile, sigma=sigma)

    plt.figure(figsize=(8,5))
    plt.plot(t, rep_profile, label="Rådata", alpha=0.5)
    plt.plot(t, smooth, label=f"sigma={sigma}", linewidth=3)
    plt.legend()
    plt.grid(True)
    plt.show()

interact(plot_sigma, sigma=(0.1, 5, 0.1))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# signal
x = rep_profile.copy()
t = np.arange(len(x))

# Gaussian kernel
sigma = 2
k = np.arange(-6, 7)
w = np.exp(-k**2 / (2*sigma**2))
w = w / w.sum()  # normaliser

# tom output
y = np.zeros_like(x)

for i in range(len(x)):
    clear_output(wait=True)

    # velg lokalt vindu
    indices = k + i
    valid = (indices >= 0) & (indices < len(x))

    k_valid = k[valid]
    indices_valid = indices[valid]
    w_valid = w[valid]

    # beregn ett punkt (konvolusjon)
    y[i] = np.sum(x[indices_valid] * w_valid)

    plt.figure(figsize=(10,6))

    # subplot 1: signal + vekt
    plt.subplot(2,1,1)
    plt.plot(t, x, label="Signal (last)")
    plt.scatter(i, x[i], color='red')

    # vis kernel plassert ved i
    plt.plot(indices_valid, w_valid * max(x), '--', label="Kernel (skalert)")

    plt.title(f"Steg i = {i}")
    plt.legend()
    plt.grid(True)

    # subplot 2: resultat
    plt.subplot(2,1,2)
    plt.plot(t, y, label="Konvolusjon (bygger opp)")
    plt.scatter(i, y[i], color='red')

    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.pause(0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# frekvensakse
w = np.linspace(0, np.pi, 500)

# Moving Average kernel (lengde 5)
N = 5
ma = np.ones(N)/N

# Fourier transform
H_ma = np.abs(np.fft.fft(ma, 500))

# Gaussian kernel
sigma = 2
k = np.arange(-10, 11)
gauss = np.exp(-k**2/(2*sigma**2))
gauss = gauss / gauss.sum()

H_gauss = np.abs(np.fft.fft(gauss, 500))

# plot
plt.plot(w, H_ma[:500], label="Moving Average")
plt.plot(w, H_gauss[:500], label="Gaussian")

plt.title("Frekvensrespons")
plt.xlabel("Frekvens")
plt.ylabel("Amplitude")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
from ipywidgets import interact

def moving_avg_demo(width=3):
    window = np.ones(width)/width
    
    y = np.convolve(rep_profile, window, mode='same')
    
    plt.plot(rep_profile, label="Rådata", alpha=0.5)
    plt.plot(y, label=f"MA width={width}", linewidth=3)
    plt.legend()
    plt.grid(True)
    plt.show()

interact(moving_avg_demo, width=(1, 10, 1))


In [ ]:
from scipy.ndimage import gaussian_filter1d

def gaussian_demo(sigma=2):
    y = gaussian_filter1d(rep_profile, sigma=sigma)

    plt.plot(rep_profile, label="Rådata", alpha=0.5)
    plt.plot(y, label=f"Gaussian σ={sigma}", linewidth=3)
    plt.legend()
    plt.grid(True)
    plt.show()

interact(gaussian_demo, sigma=(0.1, 5, 0.1))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# signal (enkelt eksempel)
t = np.linspace(0, 10, 200)
x = np.sin(t) + 0.5*np.sin(5*t)

# impulse response (Gaussian)
sigma = 0.5
tau = 0.0
k = np.linspace(-2, 2, 100)
h = np.exp(-k**2/(2*sigma**2))
h = h / np.sum(h)

# konvolusjon
y = np.convolve(x, h, mode='same')

# plot
plt.figure(figsize=(10,6))

plt.subplot(3,1,1)
plt.plot(t, x)
plt.title("Input signal x(t)")

plt.subplot(3,1,2)
plt.plot(k, h)
plt.title("Impulse response h(t) (Gaussian)")

plt.subplot(3,1,3)
plt.plot(t, y)
plt.title("Output y(t) = x * h")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 20, 500)

# cases
cases = [
    (-0.3, 2.0, "Dempet oscillasjon"),
    (0.1, 2.0, "Ustabil oscillasjon"),
    (-1.0, 0.0, "Ikke-oscillerende")
]

plt.figure()

for alpha, omega, label in cases:
    y = np.exp(alpha*t) * np.cos(omega*t)
    plt.plot(t, y, label=label)

plt.legend()
plt.title("Effekt av eigenverdier")
plt.xlabel("Tid")
plt.grid(True)
plt.show()

In [ ]:
from ipywidgets import interact
import numpy as np
import matplotlib.pyplot as plt

def eigen_demo(alpha=-0.2, omega=2.0):
    t = np.linspace(0, 20, 500)

    y = np.exp(alpha*t) * np.cos(omega*t)

    plt.figure(figsize=(8,4))
    plt.plot(t, y)
    plt.axhline(0, linestyle='--')

    plt.title(f"alpha={alpha}, omega={omega}")
    plt.xlabel("Tid")
    plt.ylabel("Respons")
    plt.grid(True)
    plt.show()

interact(eigen_demo, 
         alpha=(-1, 0.5, 0.05),
         omega=(0, 5, 0.1));

In [ ]:
from ipywidgets import interact
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 20, 1000)

def multi_mode(a1=-0.2, w1=1.0,
               a2=-0.5, w2=2.0,
               a3=-1.0, w3=0.0):

    y1 = np.exp(a1*t)*np.cos(w1*t)
    y2 = np.exp(a2*t)*np.cos(w2*t)
    y3 = np.exp(a3*t)*np.cos(w3*t)

    y = y1 + y2 + y3

    plt.figure(figsize=(8,5))
    plt.plot(t, y, label="Total respons", linewidth=3)

    plt.plot(t, y1, '--', label="Mode 1")
    plt.plot(t, y2, '--', label="Mode 2")
    plt.plot(t, y3, '--', label="Mode 3")

    plt.legend()
    plt.title("Flere oscillasjonsmodi (eigenverdier)")
    plt.grid(True)
    plt.show()

interact(multi_mode);


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t = np.linspace(0, 24, len(rep_profile))

# define modes (alpha, omega)
modes = [
    (-0.1, 0.5),   # slow inter-area mode (critical)
    (-0.3, 1.5),   # local mode
    (-1.0, 0.0)    # fast damped mode
]

dt = t[1] - t[0]

responses = []
total = np.zeros_like(t)

for alpha, omega in modes:
    # impulse response
    h = np.exp(alpha*t) * np.cos(omega*t)

    # convolution
    y = np.convolve(rep_profile, h, mode='same') * dt

    responses.append(y)
    total += y
    

In [ ]:
plt.figure(figsize=(9,6))

for i, y in enumerate(responses):
    plt.plot(t, y, '--', label=f"Mode {i+1}")

plt.plot(t, total, linewidth=3, label="Total frequency response")

plt.xlabel("Time [hours]")
plt.ylabel("Δf (p.u.)")
plt.title("Frequency response from real load input")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(9,5))

for i, y in enumerate(responses):
    alpha, omega = modes[i]

    if abs(alpha) < 0.2:
        plt.plot(t, y, linewidth=3, label=f"CRITICAL mode {i+1}")
    else:
        plt.plot(t, y, '--', alpha=0.5)

plt.title("Critical mode identification")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from ipywidgets import interact

def multi_mode_sim(a1=-0.1, w1=0.5,
                   a2=-0.3, w2=1.5,
                   a3=-1.0, w3=0.0):

    modes = [(a1,w1), (a2,w2), (a3,w3)]

    total = np.zeros_like(t)

    plt.figure(figsize=(9,5))

    for i, (alpha, omega) in enumerate(modes):
        h = np.exp(alpha*t) * np.cos(omega*t)
        y = np.convolve(rep_profile, h, mode='same') * dt

        total += y
        plt.plot(t, y, '--', label=f"Mode {i+1}")

    plt.plot(t, total, linewidth=3, label="Total")

    plt.title("Multi-mode frequency response")
    plt.legend()
    plt.grid(True)
    plt.show()

interact(multi_mode_sim,
         a1=(-0.5, 0.1, 0.02), w1=(0,2,0.1),
         a2=(-1.0, 0.1, 0.05), w2=(0,3,0.1),
         a3=(-2.0, -0.1, 0.05), w3=(0,1,0.1));

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert

# use your signal (VERY IMPORTANT)
signal = smooth - np.mean(smooth)

t = np.linspace(0, len(signal), len(signal))

# analytic signal
analytic = hilbert(signal)

# amplitude envelope
amplitude = np.abs(analytic)

# phase
phase = np.unwrap(np.angle(analytic))

# estimate frequency
omega = np.gradient(phase, t)

# estimate damping
alpha = np.gradient(np.log(amplitude + 1e-6), t)

print("Estimated frequency (mean):", np.mean(omega))
print("Estimated damping (mean):", np.mean(alpha))

In [ ]:
plt.figure(figsize=(10,5))

plt.subplot(2,1,1)
plt.plot(t, omega)
plt.title("Instantaneous frequency")

plt.subplot(2,1,2)
plt.plot(t, alpha)
plt.title("Estimated damping")

plt.tight_layout()
plt.show()

In [ ]:
fft = np.fft.fft(signal)
freqs = np.fft.fftfreq(len(signal), d=1)

plt.plot(freqs[:len(freqs)//2], np.abs(fft[:len(freqs)//2]))
plt.title("Frequency spectrum")
plt.xlabel("Frequency")
plt.ylabel("Amplitude")
plt.show()

In [ ]:
plt.scatter(omega, alpha)

plt.axhline(0, linestyle='--')
plt.xlabel("Frequency ω")
plt.ylabel("Damping α")
plt.title("Estimated eigenvalues")

plt.grid(True)
plt.show()